# TVM Optimized Tomography

## Useful Defines

In [ ]:
import numpy as np
import cupy as cp
# -------------------- frequency utilities --------------------

def _fftfreq_radius(shape, voxel_size=1.0):
    freqs = [np.fft.fftfreq(n, d=voxel_size) for n in shape]
    grids = np.meshgrid(*freqs, indexing="ij")
    r = np.sqrt(sum(g**2 for g in grids))
    return r

# -------------------- filtering & masks --------------------

def _lowpass_filter(arr, voxel_size=1.0, cutoff_cyc_per_unit=0.25, trans_width=0.0):
    """
    3D isotropic low-pass filter.
    cutoff_cyc_per_unit: frequency cutoff (cycles / unit)
    trans_width: optional cosine transition width for a soft edge
    """
    F = np.fft.fftn(arr)
    r = _fftfreq_radius(arr.shape, voxel_size)
    mask = np.zeros_like(r, dtype=np.float32)
    if trans_width > 0:
        inner = r <= (cutoff_cyc_per_unit - trans_width)
        outer = r >= (cutoff_cyc_per_unit + trans_width)
        mid = (~inner) & (~outer)
        mask[inner] = 1.0
        mask[mid] = 0.5 * (1 + np.cos(np.pi * (r[mid] - (cutoff_cyc_per_unit - trans_width)) / (2*trans_width)))
    else:
        mask[r <= cutoff_cyc_per_unit] = 1.0
    F *= mask
    return np.fft.ifftn(F).real

def _checkerboard_masks(shape):
    """Return boolean masks for even/odd checkerboard pattern in 3D."""
    idx = np.meshgrid(*[np.arange(n) for n in shape], indexing="ij")
    parity = sum(idx) % 2
    m0 = (parity == 0)
    m1 = ~m0
    return m0, m1

# -------------------- FSC core --------------------

def _fsc(vol1, vol2, voxel_size=1.0, df=0.01, eps=1e-12):
    """Compute Fourier shell correlation between two 3D arrays."""
    assert vol1.shape == vol2.shape

    vol1 = vol1 - np.mean(vol1)
    vol2 = vol2 - np.mean(vol2)

    F1 = np.fft.fftn(vol1)
    F2 = np.fft.fftn(vol2)
    r = _fftfreq_radius(vol1.shape, voxel_size)

    r_max = r.max()
    edges = np.arange(0, r_max + df, df)
    bins = np.digitize(r.ravel(), edges) - 1
    n_shells = edges.size - 1

    cross = (F1 * np.conj(F2)).ravel().real
    p1 = (np.abs(F1)**2).ravel()
    p2 = (np.abs(F2)**2).ravel()

    valid = (bins >= 0) & (bins < n_shells)
    b = bins[valid]
    num = np.bincount(b, weights=cross[valid], minlength=n_shells)
    den1 = np.bincount(b, weights=p1[valid], minlength=n_shells)
    den2 = np.bincount(b, weights=p2[valid], minlength=n_shells)

    fsc = num / (np.sqrt(den1 * den2) + eps)
    freqs = 0.5 * (edges[:-1] + edges[1:])
    return freqs, fsc

# -------------------- Checkerboard SFSC --------------------

def sfsc_checkerboard(volume, voxel_size=1.0, df=0.01, trans_width=0.0):
    """
    Self-Fourier Shell Correlation (3D checkerboard version)
    Steps:
      1) Low-pass to 0.25 / voxel_size (to avoid aliasing)
      2) Split volume into two interleaved sublattices (x+y+z mod 2)
      3) Reconstruct both halves by low-pass interpolation
      4) Compute FSC between the two reconstructed maps
    """
    vol = volume.astype(np.float32)
    vol = np.asarray(vol)
    cutoff = 0.25 / voxel_size

    # Step 1: anti-alias filter
    vol_lp = _lowpass_filter(vol, voxel_size, cutoff, trans_width)
    #vol_lp = vol
    # Step 2: checkerboard split
    m0, m1 = _checkerboard_masks(vol.shape)
    v0_sparse = np.zeros_like(vol_lp)
    v1_sparse = np.zeros_like(vol_lp)
    v0_sparse[m0] = vol_lp[m0]
    v1_sparse[m1] = vol_lp[m1]

    # Step 3: reconstruct each half to full grid
    v0_rec = _lowpass_filter(v0_sparse, voxel_size, cutoff, trans_width)
    v1_rec = _lowpass_filter(v1_sparse, voxel_size, cutoff, trans_width)

    # Step 4: FSC between reconstructed halves
    freqs, fsc = _fsc(v0_rec, v1_rec, voxel_size, df=df)
    resolution = fsc_resolution(freqs, fsc, threshold=0.143)
    if resolution is None:
        resolution = voxel_size
    return resolution, freqs, fsc, v0_rec, v1_rec

# -------------------- Resolution utility --------------------

def fsc_resolution(freqs, fsc, threshold=0.143):
    """FSC resolution with monotonicized curve and linear interpolation."""

    # Enforce monotonic non-increasing FSC (from low to high frequency)
    fsc_mono = np.minimum.accumulate(fsc)

    below = np.where(fsc_mono < threshold)[0]
    if not len(below):
        return None

    i = below[0]
    if i == 0:
        f_cut = freqs[i]
    else:
        f1, f2 = freqs[i-1], freqs[i]
        y1, y2 = fsc_mono[i-1], fsc_mono[i]
        f_cut = f1 + (threshold - y1) * (f2 - f1) / (y2 - y1)

    return 1.0 / f_cut



In [ ]:
import tomosipo as ts
import numpy as np
from tomobase.data import Volume, Sinogram
import cupy as cp
import os
def project(volume, angles):
    """
    Project a 3D volume to generate sinograms at specified angles.

    Parameters:
    - volume: 3D numpy array of shape (num_slices, height, width)
    - angles: 1D numpy array of projection angles in radians

    Returns:
    - sinograms: 3D numpy array of shape (num_angles, num_detectors, num_slices)
    """
    volume.data = volume.data.transpose(2,0,1)  # (S,H,W)->(W,S,H) 021
 

    angles_rad = np.radians(angles+90)
    vg = ts.volume(shape=(128,128,128))
    pg = ts.parallel(angles=angles_rad, shape=(128, 128))
    A = ts.operator(vg, pg)


    sinogram = A(volume.data).transpose(1,0, 2)
    return Sinogram(sinogram, angles)

def reconstruct_tvm(sinogram, vol_file=None, num_iterations=100, lambda_tv=0.1 ):
    """
    Reconstruct a 3D volume using SIRT with Total Variation regularization.
    
    Parameters:
    - sinogram: Sinogram object with projection data
    - num_iterations: Number of iterations
    - lambda_tv: TV regularization strength (higher = more smoothing)
    
    Returns:
    - volume: Reconstructed Volume object
    """
    angles_rad = np.radians(sinogram.angles + 90)
    vg = ts.volume(shape=(sinogram.data.shape[1],sinogram.data.shape[1], sinogram.data.shape[2]), size=(1, 1, 1))
    pg = ts.parallel(angles=angles_rad, shape=(sinogram.data.shape[1], sinogram.data.shape[2]), size=(1.0, 1.0))
    
    A = ts.operator(vg, pg)
    
    # SIRT weights
    R =  1 / A(np.ones(A.domain_shape))
    R = np.clip(R, a_min=None, a_max=1 / ts.epsilon)
    R = np.asarray(R)
    C = 1 / A.T(np.ones(A.range_shape))
    C = np.clip(C, a_min=None, a_max=1 / ts.epsilon)
    C = np.asarray(C)
    
    y = np.asarray(sinogram.data.transpose(1, 0, 2))
    if vol_file is None:
        x_rec = np.asarray(np.zeros(A.domain_shape, dtype=np.float32))
    else:
        x_rec = np.asarray(Volume.from_file(vol_file).data)
        num_iterations =  int(vol_file.name.split('_')[0]) 
    
    for i in range(num_iterations):
        # Standard SIRT update
        residual = y - A(x_rec)
        sirt_update = C * A.T(R * residual)
        
        # TV gradient (using finite differences)
        #tv_grad = compute_tv_gradient(x_rec)
        
        # Combined update with TV regularization
        #x_rec += sirt_update - lambda_tv * tv_grad
        x_rec += sirt_update
        
        # Non-negativity constraint (optional but common in tomography)
        x_rec = np.maximum(x_rec, 0)
    
  
    x_rec = x_rec.transpose(1, 2, 0)  # (W,S,H)->(S,H,W) 120
    return Volume(x_rec)


def compute_tv_gradient(volume):
    """
    Compute the gradient of the Total Variation functional.
    Uses isotropic TV: sum of gradient magnitudes.
    """
    # Compute gradients along each axis using finite differences
    grad_x = np.zeros_like(volume)
    grad_y = np.zeros_like(volume)
    grad_z = np.zeros_like(volume)
    
    # Forward differences
    grad_x[:-1, :, :] = volume[1:, :, :] - volume[:-1, :, :]
    grad_y[:, :-1, :] = volume[:, 1:, :] - volume[:, :-1, :]
    grad_z[:, :, :-1] = volume[:, :, 1:] - volume[:, :, :-1]
    
    # Gradient magnitude (with small epsilon for numerical stability)
    eps = 1e-8
    grad_mag = np.sqrt(grad_x**2 + grad_y**2 + grad_z**2 + eps)
    
    # Compute TV gradient (divergence of normalized gradient)
    tv_grad = np.zeros_like(volume)
    
    # Backward divergence for x
    div_x = np.zeros_like(volume)
    div_x[1:-1, :, :] = (grad_x[1:-1, :, :] / grad_mag[1:-1, :, :] - 
                          grad_x[:-2, :, :] / grad_mag[:-2, :, :])
    div_x[0, :, :] = grad_x[0, :, :] / grad_mag[0, :, :]
    div_x[-1, :, :] = -grad_x[-2, :, :] / grad_mag[-2, :, :]
    
    # Backward divergence for y
    div_y = np.zeros_like(volume)
    div_y[:, 1:-1, :] = (grad_y[:, 1:-1, :] / grad_mag[:, 1:-1, :] - 
                          grad_y[:, :-2, :] / grad_mag[:, :-2, :])
    div_y[:, 0, :] = grad_y[:, 0, :] / grad_mag[:, 0, :]
    div_y[:, -1, :] = -grad_y[:, -2, :] / grad_mag[:, -2, :]
    
    # Backward divergence for z
    div_z = np.zeros_like(volume)
    div_z[:, :, 1:-1] = (grad_z[:, :, 1:-1] / grad_mag[:, :, 1:-1] - 
                          grad_z[:, :, :-2] / grad_mag[:, :, :-2])
    div_z[:, :, 0] = grad_z[:, :, 0] / grad_mag[:, :, 0]
    div_z[:, :, -1] = -grad_z[:, :, -2] / grad_mag[:, :, -2]
    
    tv_grad = -(div_x + div_y + div_z)
    
    return tv_grad

In [ ]:
import copy 
from itertools import combinations
import copy
import os 
import gc  # Garbage collector for memory cleanup
from pathlib import Path
from tomobase import processes


def find_and_reconstruct(folder,  start, end, sino, recon, *args, **kwargs):
    halfmap = kwargs.pop('halfmap', None)
    if halfmap is None:
        matches = list(Path(folder).glob(f"*_recon_{start}_{end}.rec"))
    else:
        matches = list(Path(folder).glob(f"*_recon_{start}_{end}_{halfmap}.rec"))

    if matches:
        vol = recon(sino, vol_file=matches[0], *args, **kwargs)
        os.remove(matches[0])
    else:
        vol = recon(sino, vol_file=None, *args, **kwargs)

    if halfmap is None:
        vol.to_file(Path(folder)/f"{kwargs.get('num_iterations',100)}_recon_{start}_{end}.rec")
    else:
        vol.to_file(Path(folder)/f"{kwargs.get('num_iterations',100)}_recon_{start}_{end}_{halfmap}.rec")
    return vol

def subsection_sinogram(sino, start, end):
    sinogram = copy.deepcopy(sino)
    sinogram.data = sinogram.data[start:end,:, :]
    sinogram.angles = sinogram.angles[start:end]
    sinogram.times = sinogram.times[start:end]
    return sinogram

def sfsc_split_projection(sinogram, recon_func, folder='', start=0, end=0, voxel_size=1.0, df=0.01, **recon_kwargs):
    """
    Self-FSC using split-projection method.
    Proper for limited-angle tomography where checkerboard fails.
    
    Randomly splits projections into two halves, reconstructs each,
    then computes FSC between the two independent reconstructions.
    """
    matches = list(Path(folder).glob(f"*_recon_{start}_{end}.rec"))
    half = sinogram.data.shape[0] // 2

    # Create two sub-sinograms
    sino1 = subsection_sinogram(sinogram, 0, half)
    sino2 = subsection_sinogram(sinogram, half, sinogram.data.shape[0])

    vol1 = processes.astra_reconstruct(sino1, use_gpu=False)
    vol2 = processes.astra_reconstruct(sino2, use_gpu=False)
    
    # Compute FSC between the two independent reconstructions
    freqs, fsc = _fsc(vol1.data, vol2.data, voxel_size, df=df)
    resolution = fsc_resolution(freqs, fsc, threshold=0.143)
    
    if resolution is None:
        # If FSC never drops, return Nyquist limit
        resolution = 2 * voxel_size
    
    # Clean up large temporary volumes
    del vol1, vol2, sino1, sino2
    gc.collect()
    
    return resolution, freqs, fsc

## Generalized Process

In [ ]:

from scipy.stats import kendalltau

def get_windows(sino,sino_alt, recon, windows=None, folder='', *args, **kwargs):

    print("|------------------------------------------------|")
    print(f"Iterations: {kwargs.get('num_iterations', 100)} | Lambda TV: {kwargs.get('lambda_tv', 0.1)}")
    rolling_ave = 4
    no_windows = False
    if windows is None:
        windows = np.zeros((len(sino.times), 3), int)
    
        for i in range(len(sino.times)):
            windows[i,0] = i
            windows[i,1] = i+20
            windows[i,2] = 0
        no_windows = True
   
    for i in range(len(sino.times)):
        if i >= 1:
            if windows[i-1,1]>=len(sino.times):
                for j in range(i, windows.shape[0]):
                    windows[j,2] = 0
                break
            elif no_windows:
                windows[i,1] = windows[i-1,1]+1

        windows[i,2] = 1
        print("**------------------------------------------------**")
        print(f"Window {i}: {windows[i,0]}-{windows[i,1]}")
        break_out = False
        direction = 0
        new_direction = 0
        while not break_out:
            j_start = max(windows[i,0]+1, windows[i,1]-rolling_ave)
            j_end = min(len(sino.times), windows[i,1]+rolling_ave)

            resolutions = np.zeros((j_end - j_start, 2))
            for j in range(j_start, j_end):
                sino_subsect = subsection_sinogram(sino, windows[i, 0], j)
           
                
                resolution = sfsc_split_projection(sino_subsect, recon, folder=folder, start=windows[i, 0], end=j, voxel_size=sino.pixelsize,
                                                num_iterations=kwargs.get('num_iterations', 100),
                                                lambda_tv=kwargs.get('lambda_tv', 0.1))[0]
                #resolution = sfsc_checkerboard(vol.data, voxel_size=sino.pixelsize, df=0.01, trans_width=0.0)[0]
              
                resolutions[j - j_start, 0] = j
                resolutions[j - j_start, 1] = resolution
                del sino_subsect

            tau, p = kendalltau(resolutions[:,0], resolutions[:,1])
            change = 0
            if tau > 0:
                print(f"Window {i}: {windows[i,0]}-{windows[i,1]} | Kendall's tau: {tau:.3f} (p={p:.3e}) => Decreasing resolution, shrinking window")
                new_direction = -1
                change = -1

            elif tau < 0:
                print(f"Window {i}: {windows[i,0]}-{windows[i,1]} | Kendall's tau: {tau:.3f} (p={p:.3e}) => Increasing resolution, expanding window")
                new_direction = 1
                change = 1
            else:
                print(f"Window {i}: {windows[i,0]}-{windows[i,1]} | Kendall's tau: {tau:.3f} (p={p:.3e}) => No change in resolution")
                new_direction = 0

            if np.abs(direction - new_direction) == 2:
                break_out= True
            elif new_direction == 0 and direction == 0:
                break_out = True

            
            direction = new_direction
            if break_out:
                sino_subsect = subsection_sinogram(sino, windows[i, 0], windows[i, 1])
                sino_subsect2 = subsection_sinogram(sino_alt, windows[i, 0], windows[i, 1])
                resolution = sfsc_split_projection(sino_subsect, recon, folder=folder, start=windows[i, 0], end=windows[i, 1], voxel_size=sino.pixelsize,
                                                num_iterations=kwargs.get('num_iterations', 100),
                                                lambda_tv=kwargs.get('lambda_tv', 0.1))[0]
                vol = processes.astra_reconstruct(sino_subsect, use_gpu=False)
                vol2 = processes.astra_reconstruct(sino_subsect2, use_gpu=False)
                #resolution = sfsc_checkerboard(vol.data, voxel_size=sino.pixelsize, df=0.01, trans_width=0.0)[0]
                vol.to_file(os.path.join(folder, f"recon_mof_{windows[i,0]}_{windows[i,1]}.rec"))
                vol2.to_file(os.path.join(folder, f"recon_gold_{windows[i,0]}_{windows[i,1]}.rec"))
                del vol, vol2
                print(f"  → Optimal window found: {windows[i,0]}-{windows[i,1]} | Res: {resolution:.3f} | time: {np.mean(sino.times[windows[i,0]:windows[i,1]]):.2f} units")
                del sino_subsect, sino_subsect2
            windows[i,1] += change
            gc.collect()

    return windows

def dynamic_reconstruction(sino, sino_alt, folder, iterations=100, lambda_tv=0.1, win=None): 
    for iter_idx in range(100,iterations+1):
        # Use split-projection FSC
        full_resolution = sfsc_split_projection(sino, reconstruct_tvm, folder='', start=0, end=len(sino.times), voxel_size=sino.pixelsize,
                                               num_iterations=100, lambda_tv=lambda_tv)[0]
        #full_resolution = sfsc_checkerboard(vol.data, voxel_size=sino.pixelsize, df=0.01, trans_width=0.0)[0]
        print(f"\n{'='*60}")
        print(f"TVM Iteration {iter_idx+1}/{iterations} | Full Volume Reconstruction | Res: {full_resolution:.3f}")
        print(f"{'='*60}\n")

        if iter_idx == 0:
            win = get_windows(sino, sino_alt, reconstruct_tvm, windows=None, folder=folder, num_iterations=iter_idx+1, lambda_tv=lambda_tv)
        else:
            win = get_windows(sino, sino_alt,reconstruct_tvm, windows=None, folder=folder, num_iterations=iter_idx+1, lambda_tv=lambda_tv)
        
        #for win_idx in range(win.shape[0]):
        #    if win[win_idx,2]==1:
        #       sino_subset = subsection_sinogram(sino, win[win_idx,0], win[win_idx,1])
        #       recon = find_and_reconstruct(folder, win[win_idx,0], win[win_idx,1], sino_subset, reconstruct_tvm, num_iterations=iterations, lambda_tv=lambda_tv)
        #       filename = f"{folder}/{iter_idx}_recon_{win[win_idx,0]}_{win[win_idx,1]}.rec"
        #       matches = list(Path(folder).glob(f"*_recon_{win[win_idx,0]}_{win[win_idx,1]}.rec"))
        #       for match in matches:
        #           os.remove(match)
        #       recon.to_file(filename)
        #       print(f"Saved: {filename}")
                
                # Cleanup
        #       del sino_subset, recon
        

        gc.collect()

    # Cleanup old files
   # for file in os.listdir(folder):
    #    iteration_file = int(file.split('_')[0])
    #    if iteration_file < iterations: 
    #       path = os.path.join(folder, file)
     #       if os.path.exists(path):
    #          os.remove(path)

In [37]:
folder_name= r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\Volumes128\Rod-D-4.0'
folder_name = r'D:\dynamic_sims\moftest'
import tomobase.processes as processes
from tomobase.tiltschemes import GRS
import stackview
import pathlib
import os
import numpy as np
from tomobase.data import Sinogram, Volume


sino = Sinogram.from_file(pathlib.Path(r'C:\Users\TCraig\Pictures\Rod\DIPs\mof.mrc'))
sino = processes.image_processing.scaling.bin(sino)
#sino = processes.align_tilt_axis_rotation(sino)
#sino = processes.align_tilt_axis_shift(sino)
print(sino.data.shape)
sino2 = Sinogram.from_file(os.path.join(folder_name, 'mofonly.mrc'))
sino1 = Sinogram.from_file(os.path.join(folder_name, 'gold.mrc'))
sino.angles = sino1.angles


pad_width = (
    (0, 0),        # axis 0: no padding
    (0, 145),      # axis 1
    (0, 152),      # axis 2
)

sino.data = np.pad(sino.data, pad_width, mode="constant")
print(sino.data.shape)
shift = 106  # example
sino.data = np.roll(sino.data, shift=shift, axis=2)
shift2 = 75  # example
sino.data = np.roll(sino.data, shift=shift2, axis=1)

sino.data = sino.data.transpose(0,2,1)
sino.to_file(os.path.join(folder_name, 'mof_512.mrc'))
sino = processes.image_processing.scaling.bin(sino)
sino = processes.image_processing.scaling.bin(sino)
sino.to_file(os.path.join(folder_name, 'mof_128.mrc'))
#sino = processes.image_processing.scaling.bin(sino)
'''
tiltscheme = GRS(-70, 70,0)
angles = np.array([tiltscheme.get_angle() for i in range(100)])

for i in range( 1,101):
    vol_path = os.path.join(folder_name, f'{i}data.rec')
    vol = Volume.from_file(vol_path)
    sino_img = processes.project(vol, [angles[i-1]])
    sino_img.times = np.array([i])
    if i == 1:
        sino = copy.deepcopy(sino_img)
    else:
        sino.data = np.concatenate((sino.data, sino_img.data), axis=0)
        sino.angles = np.concatenate((sino.angles, sino_img.angles), axis=0)
        
        sino.times = np.concatenate((sino.times, sino_img.times), axis=0)
'''
import astra
astra.test()

folder_name = r'D:\dynamic_sims\moftest'
#vol = processes.astra_reconstruct(sino, use_gpu=False)
#vol.to_file(os.path.join(folder_name, 'initial_recon.rec'))
#stackview.slice(sino.data)

stackview.orthogonal(vol.data)
#dynamic_reconstruction(sino2, sino1, folder_name, iterations=100, lambda_tv=0.1)

2026-01-14 02:28:41,338 - DEBUG - ()
2026-01-14 02:28:41,338 - DEBUG - {'obj': <tomobase.data.sinogram.Sinogram object at 0x000001BD38893FD0>, 'factor': 2}


(106, 367, 360)
(106, 512, 512)


2026-01-14 02:28:43,773 - DEBUG - ()
2026-01-14 02:28:43,773 - DEBUG - {'obj': <tomobase.data.sinogram.Sinogram object at 0x000001BD38893FD0>, 'factor': 2}
2026-01-14 02:28:43,913 - DEBUG - ()
2026-01-14 02:28:43,913 - DEBUG - {'obj': <tomobase.data.sinogram.Sinogram object at 0x000001BD38893FD0>, 'factor': 2}


ASTRA Toolbox v2.2.0
Getting GPU info... GPU #0: NVIDIA GeForce RTX 2060, with 6143MB, CUDA compute capability 7.5
Testing basic CPU 2D functionality... Ok
Testing basic CUDA 2D functionality... Ok
Testing basic CUDA 3D functionality... Ok


In [13]:
sino = Sinogram.from_file(pathlib.Path(r'C:\Users\TCraig\Pictures\Rod\DIPs\mof.mrc'))
print(sino.data.shape)

(106, 734, 720)


In [58]:
import copy

thresh = 0.25
sino = Sinogram.from_file(pathlib.Path(r'C:\Users\TCraig\Pictures\Rod\DIPs\mof.mrc'))
sino = processes.image_processing.scaling.normalize(sino)
mask = np.zeros_like(sino.data)
mask[sino.data >= thresh] = 1.0
sino_gold = copy.deepcopy(sino)
sino_gold.data = sino_gold.data * mask
stackview.side_by_side(sino.data, sino_gold.data)


2026-01-04 15:22:48,979 - DEBUG - ()
2026-01-04 15:22:48,996 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000026C0FA26650>}


side_by_side


In [56]:
sino2 = Sinogram.from_file(os.path.join(folder_name, 'mofonly.mrc'))
sino1 = Sinogram.from_file(os.path.join(folder_name, 'gold.mrc'))
stackview.side_by_side(sino1.data, sino2.data)

side_by_side


In [59]:
folder_gold = r'\\ematbyname\emat\TimC\mofnu1000\gold2'

folder_mof = r'\\ematbyname\emat\TimC\mofnu1000\mof2'
folder_gain = r'\\ematbyname\emat\TimC\mofnu1000\gain2'
folder_loss = r'\\ematbyname\emat\TimC\mofnu1000\loss2'

sino2 = copy.deepcopy(sino2)
sino1 = sino_gold

for i in range(sino1.data.shape[0]-36):
    subsect_gold = subsection_sinogram(sino1, i, i+36)
    vol_gold = processes.astra_reconstruct(subsect_gold, use_gpu=False)
    if i  == 0:
        loss =np.zeros_like(vol_gold.data)
    else:
        loss = vol_gold.data - vol_old.data
    
    vol_loss = Volume(loss)
    if i == sino1.data.shape[0]-37:
        gain = np.zeros_like(vol_gold.data)
        
    else:
        del subsect_gold
        subsect_gold = subsection_sinogram(sino1, i+1, i+37)
        vol_new = processes.astra_reconstruct(subsect_gold, use_gpu=False)
        del subsect_gold
        gain = vol_gold.data -vol_new.data
    vol_gain = Volume(gain)
    vol_gain.to_file(os.path.join(folder_gain,f'{i}_recon.rec'))
    vol_loss.to_file(os.path.join(folder_loss,f'{i}_recon.rec'))
    del vol_gain, vol_loss, loss, gain
    vol_gold.to_file(os.path.join(folder_gold,f'{i}_recon.rec'))
    vol_old = vol_gold
    
    subsect_mof = subsection_sinogram(sino2, i, i+36)
    vol_mof = processes.astra_reconstruct(subsect_mof, use_gpu=False)
    vol_mof.to_file(os.path.join(folder_mof,f'{i}_recon.rec'))
    del vol_mof, subsect_mof
    

2026-01-04 15:27:50,931 - DEBUG - ()
2026-01-04 15:27:50,931 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000026C0F8E0D90>, 'method': 'sirt', 'iterations': 0, 'use_gpu': False}
2026-01-04 15:27:51,079 - INFO - Reconstructing...
2026-01-04 15:27:51,291 - INFO - Reconstruction using the SIRT algorithm on the CPU...
100%|██████████| 734/734 [02:54<00:00,  4.20it/s]
2026-01-04 15:30:49,591 - INFO - type of volume: <class 'tomobase.data.volume.Volume'>
2026-01-04 15:30:51,523 - DEBUG - ()
2026-01-04 15:30:51,525 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000026C1395A550>, 'method': 'sirt', 'iterations': 0, 'use_gpu': False}
2026-01-04 15:30:51,527 - INFO - Reconstructing...
2026-01-04 15:30:52,092 - INFO - Reconstruction using the SIRT algorithm on the CPU...
100%|██████████| 734/734 [02:54<00:00,  4.20it/s]
2026-01-04 15:33:48,000 - INFO - type of volume: <class 'tomobase.data.volume.Volume'>
d:\code\github\timedependenttomography\submodules\tomoba

In [ ]:
from skimage import io, filters
import copy
#sino.data = processes.image_processing.normalize(sino.data)

sino2 = Sinogram.from_file(os.path.join(folder_name, 'mofonly.mrc'))
sino1 = Sinogram.from_file(os.path.join(folder_name, 'gold.mrc'))
#stackview.slice(sino2.data)
rec1 = processes.astra_reconstruct_regularized(sino2, use_gpu=False, tv_every=10)
#rec2 = processes.astra_reconstruct_regularized(sino1, use_gpu=False, tv_every=100)
rec1.to_file(os.path.join(folder_name, 'initial_recon.rec'))
#rec2.to_file(os.path.join(folder_name, 'initial_gold.rec'))
stackview.orthogonal(rec2.data)

In [ ]:
sino3, offsets = processes.align_tilt_axis_shift(sino2, inplace=False,verbose_outputs=True)
sino3, angles = processes.align_tilt_axis_rotation(sino3, verbose_outputs=True)

print(offsets, angles)
stackview.side_by_side(sino3.data, sino2.data)

In [ ]:
stackview.slice(sino2.data)

In [ ]:
import stackview
folder_name= r'\\ematbyname\emat\TimC\DIPSTER-PublicationData\PData\Volumes128\Rod-D-4.0'
vol_path = os.path.join(folder_name, f'{i}data.rec')
vol = Volume.from_file(vol_path)
stackview.orthogonal(vol.data)


In [ ]:
vol_rec = reconstruct_tvm(sino, num_iterations=100, lambda_tv=0.1)
stackview.orthogonal(vol_rec.data)

## Memory Management

Run this cell if you need to manually clear variables and free memory:

In [ ]:
# Manual memory cleanup - run this cell when needed
import gc

# Option 1: Delete specific variables
# del vol, sino, resolution  # Uncomment and list variables to delete

# Option 2: Force garbage collection
gc.collect()
print("Garbage collection complete")

# Option 3: See what's taking up memory
import sys

# Get top 10 largest objects in memory
def get_size(obj):
    try:
        return sys.getsizeof(obj)
    except:
        return 0

# Show current namespace variables (comment out if slow)
# local_vars = list(locals().items())
# local_vars.sort(key=lambda x: get_size(x[1]), reverse=True)
# for name, obj in local_vars[:10]:
#     print(f"{name}: {get_size(obj) / 1024 / 1024:.2f} MB")

# Option 4: Clear everything except imports (CAREFUL!)
# %reset -f  # This clears ALL variables!

In [ ]:
import numpy as np
from scipy.fftpack import dctn, idctn
from scipy.ndimage import distance_transform_edt

def _initial_guess_nn(x, known):
    known = known.astype(bool)
    missing = ~known

    if not missing.any():
        return x.copy(), 1
    if not known.any():
        return np.zeros_like(x, dtype=float), 1

    # indices of nearest known (zeros in `missing`)
    _, inds = distance_transform_edt(missing, return_indices=True)
    y = x.copy()
    y[missing] = x[tuple(inds[:, missing])]
    return y, 1

def inpaintn(x, n=75, y0=None, m=2, missing_mask=None, relax=1.0, debug=False):
    """
    Stable N-D inpainting via DCT smoothing.

    Key stability changes vs your version:
      - uses norm='ortho' DCT/IDCT (unitary-ish, avoids growth)
      - scales data to O(1) during iterations
      - relax defaults to 1.0 (start here)
    """
    x = np.asarray(x, dtype=float)

    if missing_mask is not None:
        missing = (np.asarray(missing_mask) != 0)
    else:
        missing = ~np.isfinite(x)

    known = ~missing
    if known.all():
        return x.copy()

    # Finite working copy
    x0 = x.copy()
    x0[missing] = 0.0
    x0[~np.isfinite(x0)] = 0.0

    # ---- scale to avoid overflow ----
    # scale based on known values only
    absmax = np.max(np.abs(x0[known])) if np.any(known) else 1.0
    if absmax == 0:
        absmax = 1.0
    x0s = x0 / absmax  # scaled

    # Build Laplacian spectrum (DCT / Neumann)
    shape = x0s.shape
    Lambda = np.zeros(shape, dtype=float)
    for ax, N in enumerate(shape):
        k = np.arange(N, dtype=float)
        lam1d = 2.0 * (1.0 - np.cos(np.pi * k / N))
        rshape = [1] * x0s.ndim
        rshape[ax] = N
        Lambda += lam1d.reshape(rshape)
    Lambda = Lambda ** m

    # Initial guess
    if y0 is None:
        y, s0 = _initial_guess_nn(x0s, known)
    else:
        y = np.asarray(y0, dtype=float).copy() / absmax
        s0 = 1

    # S schedule: don’t start insanely high for big 3D volumes
    s = np.logspace(s0, -6, n)

    W = known.astype(float)

    for i in range(n):
        Gamma = 1.0 / (1.0 + s[i] * Lambda)
        rhs = W * (x0s - y) + y

        y_new = idctn(Gamma * dctn(rhs, norm="ortho"), norm="ortho")
        y = relax * y_new + (1.0 - relax) * y

        if debug and (not np.isfinite(y).all()):
            bad = np.where(~np.isfinite(y))
            print(f"Non-finite y at iter {i}. Example index:",
                  tuple(int(b[0]) for b in bad))
            break

    # restore known pixels (still scaled)
    y[known] = x0s[known]

    # unscale back to original units
    y *= absmax
    return y
